# Studio 3D AI Endpoints — Headless GPU Server (Colab T4 / L4)

Ejecuta **`black-forest-labs/FLUX.2-klein-4B`** y **`microsoft/TRELLIS.2-4B`** sin interfaz gráfica pesada, cada uno en **su propio entorno aislado** con motores nativos C++/GGML CUDA (`stable-diffusion.cpp` y `trellis.cpp`), serializados por un **Gateway GPU-Lease** para no pasar nunca los **15 GB VRAM / 12.7 GB RAM** de Colab.

In [1]:
# @title 1. Verificar GPU y clonar repo santiago96ml/studio-3d-ai-endpoints
!nvidia-smi
!apt-get -qq update && apt-get -qq install -y -o Dpkg::Options::='--force-confold' cmake ninja-build curl git >/dev/null

import os
REPO_DIR = '/content/studio-3d-ai-endpoints'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/santiago96ml/studio-3d-ai-endpoints.git /content/studio-3d-ai-endpoints
else:
    !git -C /content/studio-3d-ai-endpoints pull --ff-only

%cd /content/studio-3d-ai-endpoints

/bin/bash: line 1: nvidia-smi: command not found
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into '/content/studio-3d-ai-endpoints'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (45/45), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 45 (delta 10), reused 43 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (45/45), 28.15 KiB | 4.69 MiB/s, done.
Resolving deltas: 100% (10/10), done.
/content/studio-3d-ai-endpoints


In [ ]:
# @title 2. Instalar entornos aislados por modelo (FLUX.2-klein-4B + Hunyuan3D-2 Destilado / TRELLIS.2-4B + Gateway)
import os
import subprocess

MODE = 'flux2_hunyuan3d2_partfield'  # @param ['flux2_hunyuan3d2_partfield', 'flux2_hunyuan3d2', 'all', 'flux2_trellis2', 'hunyuan3d2', 'partfield', 'trellis2', 'flux2']
FLUX2_QUANT = 'Q4_K_M'  # @param ['Q4_K_M', 'Q5_K_M', 'Q8_0']
HUNYUAN3D_VARIANT = 'turbo'  # @param ['turbo', 'mini']
HUNYUAN3D_VRAM_MODE = 'auto'  # @param ['auto', 'resident', 'stage_offload']
PARTFIELD_VRAM_MODE = 'auto'  # @param ['auto', 'resident', 'stage_offload']
TRELLIS2_QUANT = 'q4'  # @param ['q4', 'q8']
TRELLIS2_INCLUDE_TEXTURE = '1'  # @param ['1', '0']
STUDIO_API_TOKEN = ''  # @param {type:'string'}

os.environ['FLUX2_QUANT'] = FLUX2_QUANT
os.environ['HUNYUAN3D_VARIANT'] = HUNYUAN3D_VARIANT
os.environ['HUNYUAN3D_VRAM_MODE'] = HUNYUAN3D_VRAM_MODE
os.environ['PARTFIELD_VRAM_MODE'] = PARTFIELD_VRAM_MODE
os.environ['TRELLIS2_QUANT'] = TRELLIS2_QUANT
os.environ['TRELLIS2_INCLUDE_TEXTURE'] = TRELLIS2_INCLUDE_TEXTURE
os.environ['STUDIO_API_TOKEN'] = STUDIO_API_TOKEN
os.environ['COLAB_MODE'] = MODE

if MODE in ('all', 'flux2', 'flux2_hunyuan3d2', 'flux2_hunyuan3d2_partfield', 'flux2_trellis2'):
    subprocess.run(['bash', 'scripts/setup_flux2.sh'], check=True)
if MODE in ('all', 'hunyuan3d2', 'flux2_hunyuan3d2', 'flux2_hunyuan3d2_partfield'):
    subprocess.run(['bash', 'scripts/setup_hunyuan3d2.sh'], check=True)
if MODE in ('all', 'partfield', 'flux2_hunyuan3d2_partfield'):
    subprocess.run(['bash', 'scripts/setup_partfield.sh'], check=True)
if MODE in ('all', 'trellis2', 'flux2_trellis2'):
    subprocess.run(['bash', 'scripts/setup_trellis2.sh'], check=True)
subprocess.run(['bash', 'scripts/setup_gateway.sh'], check=True)
print('✅ Entornos listos para modo:', MODE)

Using CPython 3.11.17
Creating virtual environment with seed packages at: /content/envs/flux2
 + packaging==26.4
 + pip==26.3
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source /content/envs/flux2/bin/activate
Using Python 3.11.17 environment at: /content/envs/flux2
Resolved 31 packages in 614ms
Prepared 30 packages in 602ms
Installed 30 packages in 46ms
 + annotated-doc==0.0.7
 + annotated-types==0.8.0
 + anyio==4.16.0
 + click==8.5.1
 + fastapi==0.165.0
 + filelock==4.3.1
 + fsspec==2026.9.0
 + h11==0.16.0
 + hf-xet==1.8.0
 + httpcore2==2.16.2
 + httptools==0.9.0
 + httpx2==2.16.2
 + huggingface-hub==2.4.0
 + idna==3.20
 + opentelemetry-api==1.45.2
 + pillow==13.0.0
 + pydantic==2.14.4
 + pydantic-core==2.50.5
 + python-dotenv==1.2.4
 + python-multipart==0.0.34
 + pyyaml==6.0.4
 + starlette==1.10.0
 + tqdm==4.70.1
 + truststore==0.10.5
 + typing-extensions==4.16.0
 + typing-inspection==0.4.5
 + uvicorn==0.54.0
 + uvloop==0.23.0
 + watchfiles==1.3.0
 + websockets==17.4
[flux

In [ ]:
# @title 3. Levantar endpoints + túnel Cloudflare (copiá la URL https://*.trycloudflare.com)
import os
mode = os.environ.get('COLAB_MODE', 'all')
!python3 -u scripts/start_endpoints.py --mode {mode}

In [ ]:
# @title 4. (Opcional) Prueba local rápida contra los endpoints corriendo en Colab
# Para ejecutar esto, primero corré la Celda 3 en segundo plano o probalo desde otra sesión/PC contra tu URL Cloudflare.
!curl -s http://127.0.0.1:8000/health | python3 -m json.tool